# Clone Repository and set up the Environment

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

Cloning into 'robust-eeg-models'...
remote: Enumerating objects: 3273, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 3273 (delta 51), reused 37 (delta 37), pack-reused 3214 (from 3)
Receiving objects: 100% (3273/3273), 832.23 MiB | 47.77 MiB/s, done.
Resolving deltas: 100% (1248/1248), done.
Updating files: 100% (2211/2211), done.
Filtering content: 100% (2/2), 663.03 MiB | 100.66 MiB/s, done.


In [1]:
%cd robust-eeg-models

/content/robust-eeg-models


In [3]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [ ]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna
!pip install torchattacks
!pip install advertorch
!pip install foolbox
!pip install captum

# Clean uninstall of briandecode
!pip uninstall -y braindecode

# Install latest braindecode and autoattack code from GitHub (which includes braindecode CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir
!pip install git+https://github.com/fra31/auto-attack

In [4]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.2.0
CTNet is available ✅


In [5]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # or ":16:8" if memory is tight
os.environ["PYTHONHASHSEED"] = "0"                  # optional, extra stability


In [6]:
import torch, torchattacks, foolbox, optuna, autoattack, captum
# import advertorch
import importlib, sys, pickle, json, random, time, datetime, numbers, hashlib, subprocess

import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [7]:
from datetime import datetime
from collections import defaultdict

from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, CosineAnnealingWarmRestarts

from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity

# Use to visualise the embeddings, COULD be used to visualise the explanations
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

#EEGMamba-MOE approximation
from models.eeg_mamba_fft import create_eegmamba, EEGMamba

# Loading data for training

In [80]:
#import numpy as np
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
from braindecode.preprocessing import create_windows_from_events
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset
from torch.utils.data import Subset

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = MOABBDataset(
    dataset_name='BNCI2014001', subject_ids=[1]
)

#----------------------------------------------------------------------
# After loading we preprocess

low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000



# 1) Split preprocessors
pre_noems = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),
    Preprocessor(lambda data, factor: np.multiply(data, factor), factor=1e6),  # V→µV
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),
]
pre_ems = [
    Preprocessor(exponential_moving_standardize, factor_new=factor_new, init_block_size=init_block_size),
]

# ---- PASS 1: pre-EMS (to get physical stats) ----
preprocess(dataset, pre_noems, n_jobs=-1)

#-----------------------------------------------------------------------

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_pre = create_windows_from_events(
    dataset,
    trial_start_offset_samples = int(-0.5 * sfreq) , # -0.5s before cue
    trial_stop_offset_samples = 0,   # 4.0s after cue (t=2s to t=6s)
    preload=True,
    # verbose=0
)

# ----------------------------------------------------------------------
# Split into train and test
splitted = windows_pre.split("session")
train_set_pre = splitted["0train"]  # Session train
test_set_pre = splitted["1test"]  # Session evaluation


X_train_pre = np.stack([train_set_pre[i][0] for i in range(len(train_set_pre))])  # (N,C,T)
train_mean_pre = X_train_pre.mean(axis=(0, 2), keepdims=True)          # (1,C,1)
train_std_pre  = X_train_pre.std(axis=(0, 2), keepdims=True) + 1e-6    # (1,C,1)
train_min_pre  = X_train_pre.min(axis=(0, 2), keepdims=True)
train_max_pre  = X_train_pre.max(axis=(0, 2), keepdims=True)

# Save prenorm stats + channel names for later mapping/clamping
ch_names_pre = windows_pre.datasets[0].windows.info["ch_names"]
os.makedirs("results", exist_ok=True)
np.savez(f"results/S{1}_prenorm_stats.npz",  # <-- replace 1 with your subject id var
         mean=train_mean_pre, std=train_std_pre,
         vmin=train_min_pre, vmax=train_max_pre,
         ch_names=np.array(ch_names_pre))

# ---- PASS 2: append EMS and rebuild windows for training/eval ----
preprocess(dataset, pre_ems, n_jobs=-1)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=int(-0.5 * sfreq),
    trial_stop_offset_samples=0,
    preload=True,
)

# now continue exactly as you already do, but on windows_dataset:
splitted = windows_dataset.split("session")
train_set = splitted["0train"]
test_set  = splitted["1test"]


# Build simple tensors to compute stats on train windows only
X_train = SliceDataset(train_set, idx=0)

X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T
y_train = np.array([y for y in SliceDataset(train_set, idx=1)])

X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
y_test = np.array(test_set.get_metadata().target)                   # (N,)

train_indices, val_indices = train_test_split(
      X_train.indices_, test_size=0.2, shuffle=False
  )
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)


# 3) materialize test tensors (NOT SliceDataset)
X = torch.tensor(X_test, dtype=torch.float32, device=device)
y = torch.tensor(y_test, dtype=torch.long, device=device)

x, y , meta = train_set[0]
print(type(x), x.shape, x.mean(), x.std())




# # Save these via _save_run(... train_mean=train_mean, train_std=train_std, train_min=train_min, train_max=train_max)


/usr/local/lib/python3.12/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
(1, 22, 1)
(1, 22, 1)
(1, 22, 1)
(1, 22, 1)
[[[-0.00398651]
  [-0.00430852]


Check if this loads as train_set = x, y or x, y , meta

---
Useful for later

In [21]:
sample = train_set[0]
print(type(sample))
print(len(sample))   # see what fields exist ()
channel_names = dataset.datasets[0].raw.info['ch_names']
print(channel_names)

<class 'tuple'>
3
['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz', 'P2', 'POz']


## Set loading functions

In [99]:
def load_subject_data_cached(dataset, subject_id):
    cache_file = f'cache/{dataset}_S{subject_id}_l{low_cut_hz}_h{high_cut_hz}_ems{init_block_size}_{factor_new}.pkl'

    if os.path.exists(cache_file):
        # Load from cache - instant!
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    # Process and cache
    train_set, test_set, train_subset, val_subset , adv = load_subject_data(dataset,subject_id)

    os.makedirs('cache', exist_ok=True)
    with open(cache_file, 'wb') as f:
        pickle.dump((train_set, test_set, train_subset, val_subset, adv), f)


    return train_set, test_set, train_subset, val_subset, adv

def load_subject_data(dataset, subject_id):

    import numpy as np
    from braindecode.datasets import MOABBDataset
    from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
    from braindecode.preprocessing import create_windows_from_events
    from sklearn.model_selection import train_test_split
    from skorch.helper import SliceDataset
    from torch.utils.data import Subset

    dataset = MOABBDataset(
        dataset_name=dataset, subject_ids=[subject_id]
    )

    #----------------------------------------------------------------------
    # After loading we preprocess

    low_cut_hz = 4.0  # low cut frequency for filtering
    high_cut_hz = 38.0  # high cut frequency for filtering
    # Parameters for exponential moving standardization
    factor_new = 1e-3
    init_block_size = 750


    # 1) Split preprocessors
    pre_noems = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),
        Preprocessor(lambda data, factor: np.multiply(data, factor), factor=1e6),  # V→µV
        Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),
    ]
    pre_ems = [
        Preprocessor(exponential_moving_standardize, factor_new=factor_new, init_block_size=init_block_size),
    ]

    # ---- PASS 1: pre-EMS (to get physical stats) ----
    preprocess(dataset, pre_noems, n_jobs=-1)

    #-----------------------------------------------------------------------

    trial_start_offset_seconds = -0.5
    # Extract sampling frequency, check that they are same in all datasets
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
    # Calculate the window start offset in samples.
    trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

    # Create windows using braindecode function for this. It needs parameters to
    # define how windows should be used.
    windows_pre = create_windows_from_events(
        dataset,
        trial_start_offset_samples = int(-0.5 * sfreq) , # -0.5s before cue
        trial_stop_offset_samples = 0,   # 4.0s after cue (t=2s to t=6s)
        preload=True,
        # verbose=0
    )

    # ----------------------------------------------------------------------
    # Split into train and test
    splitted = windows_pre.split("session")
    train_set_pre = splitted["0train"]  # Session train
    test_set_pre = splitted["1test"]  # Session evaluation

    #Calculate teh train mean, std, min and max
    X_train_pre = np.stack([train_set_pre[i][0] for i in range(len(train_set_pre))])  # (N,C,T)
    train_mean_pre = X_train_pre.mean(axis=(0, 2), keepdims=True)          # (1,C,1)
    train_std_pre  = X_train_pre.std(axis=(0, 2), keepdims=True) + 1e-6    # (1,C,1)
    train_min_pre  = X_train_pre.min(axis=(0, 2), keepdims=True)
    train_max_pre  = X_train_pre.max(axis=(0, 2), keepdims=True)

    # Save prenorm stats + channel names for later mapping/clamping
    ch_names_pre = train_set_pre.datasets[0].raw.info['ch_names']
    os.makedirs("results", exist_ok=True)
    np.savez(f"results/S{subject_id}_prenorm_stats.npz",  # <-- replace 1 with your subject id var
            mean=train_mean_pre, std=train_std_pre,
            vmin=train_min_pre, vmax=train_max_pre,
            ch_names=np.array(ch_names_pre))

    # ---- PASS 2: append EMS and rebuild windows for training/eval ----
    preprocess(dataset, pre_ems, n_jobs=-1)

    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=int(-0.5 * sfreq),
        trial_stop_offset_samples=0,
        preload=True,
    )

    # now continue exactly as you already do, but on windows_dataset:
    splitted = windows_dataset.split("session")
    train_set = splitted["0train"]
    test_set  = splitted["1test"]


    # Build simple tensors to compute stats on train windows only
    X_train = SliceDataset(train_set, idx=0)

    y_train = np.array([y for y in SliceDataset(train_set, idx=1)])

    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
    y_test = np.array(test_set.get_metadata().target)                   # (N,)

    train_indices, val_indices = train_test_split(
          X_train.indices_, test_size=0.2, shuffle=False
      )
    train_subset = Subset(train_set, train_indices)
    val_subset = Subset(train_set, val_indices)
    X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T

    adv = (train_mean_pre, train_std_pre, train_min_pre, train_max_pre)
    return train_set, test_set, train_subset, val_subset, adv


## Data loading sanity check

In [97]:
# Run once or sanity check
subject_id = 1
train_set, test_set, train_subset, val_subset, adv = load_subject_data_cached("BNCI2014001", subject_id)
train_mean_pre, train_std_pre, train_min_pre, train_max_pre = adv


print(f"Window shape: {windows_dataset[0][0].shape}")
train_subset[0][0].shape[0]
channel_names = test_set.datasets[0].raw.info['ch_names']
print(channel_names)

/usr/local/lib/python3.12/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']

## Load and inspect model

In [13]:
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.12/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


# Training

## Set model hyper params

In [ ]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}
alt_params = {'lr': 1e-3, 'batch_size': 128, 'weight_decay': 5e-3, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

## Single run mode

Single training run for model debugging, sanity checks and testing non braindecode architectures (EEGMammba)



In [7]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

device = "cuda" if torch.cuda.is_available() else "cpu"


# Load tehe training data
train_set, test_set, train_subset, val_subset= load_subject_data_cached("BNCI2014001", 1)

# Build transforms list
transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]


# Extract model params from dataset, initialise model and set hyper-parameters
classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
n_classes = len(classes)
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,


)

# Hyper params
params = mamba_params

# Toggle this to enable mamba (For EEGMamba only)
# model.enable_moe(True)

# Enable MoE head instead of standard classifier (For EEGMamba only)
# model.use_moe = False

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def make_scheduler(optimizer, last_epoch=-1):
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=10, last_epoch=last_epoch)
    cosine = CosineAnnealingLR(optimizer, T_max=n_epochs-10, last_epoch=last_epoch)
    return CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    # iterator_train=AugmentedDataLoader,
    # iterator_train__transforms=transforms,
    # iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],

    # Looking at subjects
    # optimizer = torch.optim.SGD,
    # optimizer__momentum=0.9,

    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=[
        "accuracy",
        # ("lr_scheduler", LRScheduler(make_scheduler))
        # ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=200, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=500,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

import numbers


NameError: name 'torch' is not defined

# Record Baselines for chosen models

In [32]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)  # PyTorch 1.11+
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_environment_fingerprint():
    pip_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode()
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return {
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cudnn_version": torch.backends.cudnn.version(),
        "pip_freeze": pip_freeze.splitlines(),
    }

def tiny_json(base, model, id, seed, skorch_params, backend, notes):

  os.makedirs(base, exist_ok=True)
  tiny = {
      "model_name": model,
      "subject_id": id,
      "seed": seed,
      "bandpass": {"l_freq": 4.0, "h_freq": 38.0},
      "unit_scale_to_uV": 1e6,
      "ems": {"factor_new": 1e-3, "init_block_size": 750},
      "trial_start_offset_seconds": -0.5,
      "windowing": "create_windows_from_events(session split: 0train/1test)",
      "zscore_applied": False,
      "skorch_params": skorch_params,
      "backend":backend,
      "notes": notes
  }
  return tiny

def safe_model_config(model_config: dict) -> dict:
    """Convert model_config into a JSON-serializable dict."""
    safe_cfg = {}
    for k, v in model_config.items():
        if k == "model_class":
            # store full module path + class name
            safe_cfg[k] = f"{v.__module__}.{v.__name__}" if hasattr(v, "__module__") else str(v)
        elif k == "training":
            safe_training = {}
            for tk, tv in v.items():
                if tk == "optimizer":
                    # also store optimizer class name
                    safe_training[tk] = f"{tv.__module__}.{tv.__name__}" if hasattr(tv, "__module__") else str(tv)
                else:
                    # assume JSON-friendly scalar
                    safe_training[tk] = tv
            safe_cfg[k] = safe_training
        else:
            safe_cfg[k] = v if isinstance(v, (int, float, str, bool, type(None))) else str(v)
    return safe_cfg



In [17]:
def train_single_run(model_name, subject_id, seed, dataset):

    # 0. RNG reproducibility ---------------------------------------------------

    set_all_seeds(seed)
    rng_state_np    = np.random.get_state()
    rng_state_torch = torch.get_rng_state()
    env_fp          = get_environment_fingerprint()

    print(f"\n=== Processing Subject {subject_id} for seed {seed} ===")

    # 1. data ------------------------------------------------------------------
    # Load data
    train_set, test_set, train_subset, val_subset, adv  = load_subject_data_cached(dataset, subject_id)

    # Build simple tensors to compute stats on train windows only
    X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T)
    train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
    train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

    # Empirical bounds for later clipping in attack
    train_min = X_train.min(axis=(0,2), keepdims=True)
    train_max = X_train.max(axis=(0,2), keepdims=True)

    # expose indices explicitly
    train_idx = train_subset.indices
    val_idx   = val_subset.indices
    test_idx  = np.arange(len(test_set))

    # 2. model & config --------------------------------------------------------

    # Get model config
    config = MODEL_CONFIGS[model_name]

    # Extract model params from dataset, initialise model and set hyper-parameters
    # classes needed for clf
    classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
    n_classes = len(classes)
    n_channels = train_subset[0][0].shape[0]
    n_times = train_subset[0][0].shape[1]

    model = config['model_class'](
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
    )

    # Special handling for EEGMamba
    if model_name == 'EEGMamba':
        model.enable_moe(False)  # Use standard classifier for baseline. Paper explicitly removes moe modules for
                                # single use mamba
        # Enable MoE head instead of standard classifier (For multi-use only)
        model.use_moe = False

    # 4. fit -------------------------------------------------------------------

    # Create new classifier with best parameters
    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        train_split=predefined_split(val_subset),  # Use all training data
        optimizer=config['training']['optimizer'],
        optimizer__lr=config['training']['lr'],
        optimizer__weight_decay=config['training']['weight_decay'],
        batch_size=config['training']['batch_size'],
        callbacks=["accuracy"],
        device=device,
        classes=classes,
        max_epochs=500,
    )

    # Train on full training set
    clf.fit(train_subset, y=None)

    # 5. test accuracy ---------------------------------------------------------

    # Evaluate the model after training
    y_test = test_set.get_metadata().target
    test_accuracy = clf.score(test_set, y = y_test)

    # 6. save everything -------------------------------------------------------

    _save_run(model_name, subject_id, seed,
              clf, test_set, rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean, train_std, train_min, train_max,
              config, env_fp)

    return test_accuracy

# ==============================================================================

# Generate final baseline table
def create_baseline_table(results):
    """Create a nice table of baselines"""
    rows = []

    for model_name in results.keys():
        for subject_id in subjects:
            scores = results[model_name][subject_id]
            valid_scores = [s for s in scores if not np.isnan(s)]

            if valid_scores:
                mean_acc = np.mean(valid_scores)
                std_acc = np.std(valid_scores)
                n_valid = len(valid_scores)
            else:
                mean_acc = std_acc = n_valid = np.nan

            rows.append({
                'Model': model_name,
                'Subject': subject_id,
                'Mean_Accuracy': mean_acc,
                'Std_Accuracy': std_acc,
                'N_Valid_Runs': n_valid,
                'Individual_Scores': scores
            })

    return pd.DataFrame(rows)


In [18]:
def _save_run(model_name, subject_id, seed, clf, test_set,
              rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean=None, train_std=None,
              train_min=None, train_max=None,
              model_config: dict = None,
              env_fingerprint: dict = None,
              device: str=device):
    """
    clf          : fitted skorch net (clf.module_ is torch.nn.Module)
    test_set     : braindecode Dataset (getitem -> (x, y))
    *_idx        : np.ndarray of ints
    train_mean/std/min/max : arrays shaped (C,) or (1,C,1) (we'll serialize as lists)
    model_config : dict of arch + training hyperparams
    env_fingerprint : dict from get_environment_fingerprint()
    """

    base = f"{SAVE_DIR}/{model_name}/{model_name}_S{subject_id}_seed{seed}"
    os.makedirs(base, exist_ok=True)

    # ---- 0) Metadata header ----
    meta = {
        "model_name": model_name,
        "subject_id": int(subject_id),
        "seed": int(seed),
    }

    if model_config is not None:
        meta["model_config"] = safe_model_config(model_config)
    if env_fingerprint is not None:
        meta["environment"] = env_fingerprint
    json.dump(meta, open(f"{base}/meta.json", "w"), indent=2)

    # ---- 1) Checkpoint (state_dict + optimizer) ----
    torch.save({
        "state_dict": clf.module_.state_dict(),
        "optimizer": getattr(clf, "optimizer_", None).state_dict() if hasattr(clf, "optimizer_") else None,
        "seed": seed
    }, f"{base}/checkpoint.pth")

    # ---- 2) Training curves/history ----
    hist = clf.history_
    # Adjust keys if needed:
    train_acc = [e.get("train_accuracy", e.get("train_acc")) for e in hist]
    val_acc   = [e.get("valid_accuracy", e.get("val_acc")) for e in hist]
    train_loss= [e.get("train_loss") for e in hist]
    val_loss  = [e.get("valid_loss", e.get("val_loss")) for e in hist]
    curves = {"train_acc": train_acc, "val_acc": val_acc,
              "train_loss": train_loss, "val_loss": val_loss}
    json.dump(curves, open(f"{base}/curves.json", "w"), indent=2)

    # ---- 3) Test logits (CLEAN) + loss vector ----
    # Build X_test, y_test from the Dataset (no shuffling!)
    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
    y_test = np.array(test_set.get_metadata().target)                  # (N,)
    clf.module_.eval().to(device)
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
        logits_t = clf.infer(X_test_t)   # shape (N, num_classes)
        logits = logits_t.detach().cpu().numpy()
    np.save(f"{base}/test_logits_clean.npy", logits)
    np.save(f"{base}/y_test.npy", y_test)

    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    y_test_t = torch.tensor(y_test, device=device, dtype=torch.long)
    logits_ten = torch.tensor(logits, device=device, dtype=torch.float32)
    loss_vec = loss_fn(logits_ten, y_test_t).cpu().numpy()
    np.save(f"{base}/test_loss_vector.npy", loss_vec)

    # ---- 4) RNG states ----
    pickle.dump({"numpy": rng_state_np, "torch": rng_state_torch}, open(f"{base}/rng_state.pkl", "wb"))

    # ---- 5) Splits ----
    splits = {"train_idx": train_idx.tolist(),
              "val_idx":   val_idx.tolist(),
              "test_idx":  test_idx.tolist()}
    json.dump(splits, open(f"{base}/splits.json", "w"), indent=2)

    # ---- 6) Preprocessing statistics (per-channel) ----
    prep = {"zscore_applied": False}
    if train_mean is not None:
        prep["train_mean"] = np.array(train_mean).reshape(-1).tolist()
    if train_std is not None:
        prep["train_std"]  = np.array(train_std).reshape(-1).tolist()
    if train_min is not None:
        prep["train_min"]  = np.array(train_min).reshape(-1).tolist()
    if train_max is not None:
        prep["train_max"]  = np.array(train_max).reshape(-1).tolist()
    json.dump(prep, open(f"{base}/preprocessing.json", "w"), indent=2)

    # ---- 7) Attack metadata (placeholder file to append later) ----
    # You will fill this AFTER you run attacks; we create an empty schema now for consistency.
    attack_meta = {
        "whitebox": {},
        "blackbox": {}
    }
    json.dump(attack_meta, open(f"{base}/attack_metadata.json", "w"), indent=2)

    # ---- 8) README for the run folder ----
    with open(f"{base}/README.txt", "w") as f:
        f.write(
            "Artifacts:\n"
            "- checkpoint.pth: model+optimizer state_dict\n"
            "- curves.json: train/val accuracy/loss per epoch\n"
            "- test_logits_clean.npy: logits on test set (clean)\n"
            "- test_loss_vector.npy: per-sample CE loss on test set (clean)\n"
            "- y_test.npy: test labels\n"
            "- rng_state.pkl: RNG snapshots (numpy/torch)\n"
            "- splits.json: train/val/test indices (no leakage)\n"
            "- preprocessing.json: channelwise stats\n"
            "- attack_metadata.json: to be populated after attacks\n"
            "- meta.json: model/subject/seed, environment fingerprint\n"
        )

    # ---------------------------
    # Tiny JSON: per-run manifest
    # ---------------------------
    # grab skorch hyperparams (safe dict)
    try:
        skorch_params = {}
        for k, v in clf.get_params().items():
            if isinstance(v, (str, bool)):
                skorch_params[k] = v
            elif isinstance(v, numbers.Number):
                skorch_params[k] = float(v) if isinstance(v, float) else int(v)
    except Exception:
        skorch_params = {}

    # attach environment + backend determinism fingerprint
    backend = {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if hasattr(torch.version, "cuda") else None,
        "cudnn_version": torch.backends.cudnn.version(),
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    }

    # small cache manifest (see function below)

    tiny = tiny_json(
        base, model_name, subject_id, seed, skorch_params, backend,
        notes="baseline training run"
    )

    with open(f"{base}/tiny.json", "w") as f:
        json.dump(tiny, f, indent=2)


In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------------------------------------------------
# Baseline Run
# ----------------------------------------------------------------------------------------------------------------

seeds = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

datasets = {
    "BNCIv2": ("BNCI2014001", 9),

    # Ended up not useing DEAP
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Define your models and their hyperparameters
MODEL_CONFIGS = {
    'EEGNet': {
        'model_class': EEGNetv4,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'DeepConvNet': {
        'model_class': Deep4Net,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'CTNet': {
        'model_class': CTNet,
        'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    }
}

# Store all results: results[model_name][subject_id] = [acc1, acc2, acc3, acc4, acc5]
all_results = defaultdict(lambda: defaultdict(list))


# Main loop
"""
Uncomment Main loop to run and save baslines
"""

# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{'='*60}")
#     print(f"RUNNING BASELINE FOR {model_name.upper()}")
#     print(f"{'='*60}")

#     for subject_id in subjects:
#         print(f"\n--- Subject {subject_id} ---")

#         subject_scores = []
#         for seed in seeds:
#             print(f"  Seed {seed}: RUNNING")
#             try:
#                 accuracy = train_single_run(model_name, subject_id, seed, dataset)
#                 subject_scores.append(accuracy)
#                 print(f"  Seed {seed}: {accuracy:.4f}")
#             except Exception as e:
#                 print(f"  Seed {seed}: FAILED ({e})")
#                 subject_scores.append(np.nan)

#         # Store results for this (model, subject) pair
#         all_results[model_name][subject_id] = subject_scores

#         # Calculate stats for this subject
#         valid_scores = [s for s in subject_scores if not np.isnan(s)]
#         if valid_scores:
#             mean_acc = np.mean(valid_scores)
#             std_acc = np.std(valid_scores)
#             print(f"  Subject {subject_id} baseline: {mean_acc:.4f} ± {std_acc:.4f}")
#         else:
#             print(f"  Subject {subject_id}: ALL RUNS FAILED")

# # Create and display results
# baseline_df = create_baseline_table(all_results)
# print(f"\n{'='*80}")
# print("FINAL BASELINE RESULTS")
# print(f"{'='*80}")

# # Subject-wise baselines
# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{model_name}:")
#     model_data = baseline_df[baseline_df['Model'] == model_name]

#     subject_means = []
#     for _, row in model_data.iterrows():
#         if not np.isnan(row['Mean_Accuracy']):
#             print(f"  Subject {row['Subject']}: {row['Mean_Accuracy']:.4f} ± {row['Std_Accuracy']:.4f}")
#             subject_means.append(row['Mean_Accuracy'])
#         else:
#             print(f"  Subject {row['Subject']}: FAILED")

#     # Dataset-wide average
#     if subject_means:
#         dataset_mean = np.mean(subject_means)
#         dataset_std = np.std(subject_means)
#         print(f"  → Dataset average: {dataset_mean:.4f} ± {dataset_std:.4f}")
#     else:
#         print(f"  → Dataset average: FAILED")

# # Save results
# baseline_df.to_csv('baseline_results.csv', index=False)
# print(f"\nResults saved to baseline_results.csv")

'\nUncomment Main loop to run and save baslines\n'

# Adversarial attacks


## Initialize model and load checkpoints

In [103]:
# 0) tensors and stats
def load_adv_test_data(subject_id):

  train_set, test_set, train_subset, val_subset, adv = load_subject_data_cached("BNCI2014_001", subject_id)
  train_mean_pre, train_std_pre, train_min_pre, train_max_pre = adv

  X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T
  y_test = np.array(test_set.get_metadata().target)                   # (N,)



  X = torch.tensor(X_test, dtype=torch.float32, device=device)  # (N,C,T)
  y = torch.tensor(y_test, dtype=torch.long, device=device)

  CH_NAMES = train_set.datasets[0].raw.info['ch_names']

  train_min_t = torch.tensor(train_min_pre, dtype=torch.float32, device=device)  # (1,C,1)
  train_max_t = torch.tensor(train_max_pre, dtype=torch.float32, device=device)
  train_std_np = np.array(train_std_pre).reshape(-1)  # (C,)

  return X, y, train_min_t, train_max_t, train_std_np, CH_NAMES

X, y, train_min_t, train_max_t, train_std_np, CH_NAMES = load_adv_test_data(1)

print(X.shape)
print(y.shape)
print(train_min_t.shape)
print(train_max_t.shape)
print(train_std_np.shape)

n_channels = int(X.shape[1])
n_times    = int(X.shape[2])
n_classes  = int(y.max().item() + 1)  # assumes 0..K-1 labels
print(n_channels)
print(n_times)
print(n_classes)



torch.Size([288, 22, 1125])
torch.Size([288])
torch.Size([1, 22, 1])
torch.Size([1, 22, 1])
(22,)
22
1125
4


In [55]:

from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

# Print original CTNet keys
# print(model.state_dict().keys())
path = torch.load('/content/robust-eeg-models/results/CTNet/CTNet_S1_seed123/checkpoint.pth')
print(path.keys())
model.load_state_dict(path['state_dict'])   # <- no .eval() here
model.eval()


X, y, train_min_t, train_max_t, train_std_np, CH_NAMES  = load_adv_test_data(1)

ROI_IDX = [CH_NAMES.index(n) for n in ("C3","C4","Cz") if n in CH_NAMES]
def _needs_4d_input(model, Xsample):
    try:
        with torch.no_grad():
            model(Xsample[:1])  # try 3D (N,C,T)
        print("im false")
        return False
    except Exception:
        with torch.no_grad():
            model(Xsample[:1].unsqueeze(1))  # try 4D (N,1,C,T)
        return True
NEEDS_4D = _needs_4d_input(model, X)
print(CH_NAMES)
print(ROI_IDX)
print(len(CH_NAMES))

/usr/local/lib/python3.12/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


dict_keys(['state_dict', 'optimizer', 'seed'])
im false
['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz', 'P2', 'POz']
[7, 11, 9]
22


## Initialise variables

In [ ]:
# eps_grid = [0.01, 0.02, 0.03, 0.05]
# batch_size = 128
# steps = 40
# alpha = eps/8
# restarts=5

## Set up, helper methods for adversarial attacks and eval mode

In [72]:
# ========= Cell 1: Metrics & ROI helpers =========
import numpy as np
import torch
from scipy.stats import spearmanr

# ----- channel-wise score: mean |E| over time -----
def channel_scores(E):  # (N,C,T) -> (N,C)
    return E.abs().mean(dim=-1).detach().cpu().numpy()

def spearman_ch(Ec, Ea):
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i], sa[i]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

# ----- ROI helpers -----
MOTOR_ROI = ["C3","C4","Cz"]
CH_NAMES = None  # will be set in setup_subject()

ROI_IDX = [CH_NAMES.index(n) for n in MOTOR_ROI if n in CH_NAMES] if 'CH_NAMES' in globals() and CH_NAMES else None

def roi_share(E, roi_idx=ROI_IDX):
    if roi_idx is None: return float("nan")
    sc = channel_scores(E).mean(axis=0)  # (C,)
    total = sc.sum() + 1e-12
    return float(sc[roi_idx].sum() / total)

def roi_spearman(Ec, Ea, roi_idx=ROI_IDX):
    if roi_idx is None: return float("nan")
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i, roi_idx], sa[i, roi_idx]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

def laterality_index(E, left="C3", right="C4"):
    if 'CH_NAMES' not in globals() or not CH_NAMES: return float("nan")
    if left not in CH_NAMES or right not in CH_NAMES: return float("nan")
    li, ri = CH_NAMES.index(left), CH_NAMES.index(right)
    sc = channel_scores(E).mean(axis=0)  # (C,)
    num = sc[li] - sc[ri]
    den = sc[li] + sc[ri] + 1e-8
    return float(num/den)

# ----- Top-k helpers (for report) -----
def _topk_indices_from_E(E, k=5):
    sc_mean = channel_scores(E).mean(axis=0)  # (C,)
    idx = np.argsort(sc_mean)[::-1][:k]
    return idx, sc_mean

def topk_channels(E, k=5, ch_names=CH_NAMES):
    idx, _ = _topk_indices_from_E(E, k=k)
    if ch_names:
        return [ch_names[i] for i in idx]
    return idx.tolist()

def topk_roi_share(E, k=5, roi_idx=ROI_IDX):
    if roi_idx is None: return float("nan")
    idx, _ = _topk_indices_from_E(E, k=k)
    return float(len(set(idx.tolist()).intersection(set(roi_idx))) / max(1,k))


In [73]:
# ========= Cell 2: Attacks, IG explainer, and sweep runners =========
import numpy as np, torch, torch.nn.functional as F
import torchattacks as ta
import captum.attr as CA

# ---------------- utilities ----------------
def per_channel_clamp(x, vmin, vmax):
    return torch.max(torch.min(x, vmax), vmin)

def snr_db(x, x_adv):
    d = x_adv - x
    num = x.pow(2).sum((1,2)).sqrt()
    den = d.pow(2).sum((1,2)).sqrt().clamp_min(1e-12)
    return (20.0 * torch.log10(num/den)).detach().cpu().numpy()

def per_class_acc(y_true, y_pred, nclass=None):
    yt = y_true.detach().cpu().numpy(); yp = y_pred.detach().cpu().numpy()
    K = int(nclass) if nclass is not None else int(max(yp.max(), yt.max()) + 1)
    out=[]
    for c in range(K):
        idx = (yt == c)
        out.append(float((yp[idx] == c).mean()) if idx.any() else float("nan"))
    return out

def smooth_delta_gauss(delta_t: torch.Tensor, sigma_t: float) -> torch.Tensor:
    # delta_t: (N,C,T). Gaussian LP along time using grouped conv1d (fast, on-device).
    if sigma_t is None or sigma_t <= 0: return delta_t
    device = delta_t.device
    radius = int(4 * sigma_t + 0.5)
    x = torch.arange(-radius, radius + 1, dtype=delta_t.dtype, device=device)
    kernel = torch.exp(-0.5 * (x / sigma_t).pow(2))
    kernel /= kernel.sum()
    C = delta_t.size(1)
    kernel = kernel.expand(C, 1, -1)        # (C,1,K)
    return F.conv1d(delta_t, kernel, padding=radius, groups=C)

# ---------------- μV budgets & mapping (use PRENORM stats) ----------------
# Will be set in setup_subject(): prenorm_std_np, median_std_pre
muV_grid = [0.25, 0.5, 1.0, 2.0]  # report-ready grid
def eps_from_muV(mu_v):   # scalar ε in z-space
    return float(mu_v / median_std_pre)

def eps_uV_per_channel(eps_z):  # per-channel μV implied by εᶻ
    return (eps_z * prenorm_std_np).tolist()

# ---------------- IG explainer ----------------
def pick_subset(X_in, y_in, max_n=None):
    if max_n is None: max_n = ATTR_MAX_N
    if max_n and X_in.size(0) > max_n:
        return X_in[:max_n], y_in[:max_n]
    return X_in, y_in

def attr_IG(X_in, y_in):
    Xi, yi = pick_subset(X_in, y_in)
    Xi = Xi.detach().clone().requires_grad_(True)
    # internal_batch_size must be >= #examples to avoid Captum warning
    internal_bs = int(Xi.size(0))
    return ig.attribute(
        Xi, target=yi, n_steps=N_STEPS_IG,
        baselines=torch.zeros_like(Xi),
        internal_batch_size=internal_bs
    )

# single registry (we dropped LRP)
EXPLAINERS = {"IG": attr_IG}

# ---------------- L_inf sweep (FGSM/PGD) ----------------
def run_linf_sweep(atk_name, eps_list_muV, steps=None, alpha_rule=lambda e: e/8,
                   batch=128, seed=42, lp_sigma_t=None, cap_idx=None, E_CLEAN=None):
    global model, X, y, train_min_t, train_max_t, prenorm_std_np

    ctor = {"FGSM": ta.FGSM, "PGD": ta.PGD}[atk_name]
    rows=[]
    N = X.size(0)

    # Prepare subset accumulator
    start = cap_idx.start or 0
    stop  = cap_idx.stop
    cap_N = stop - start
    adv_subset = []

    with torch.no_grad():
        num_classes = int(model(X[:1]).shape[-1])
        preds_clean = model(X).argmax(1).cpu()
        clean_acc = float((preds_clean == y.cpu()).float().mean().item())
        clean_acc_pc = per_class_acc(y.cpu(), preds_clean, num_classes)

    for mu_v in eps_list_muV:
        eps_z = eps_from_muV(mu_v)
        kwargs = {}
        if atk_name == "PGD":
            kwargs["steps"] = steps if steps is not None else 40
            kwargs["alpha"] = float(alpha_rule(eps_z))
            kwargs["random_start"] = True

        atk = ctor(model, eps=float(eps_z), **kwargs)

        preds_adv=[]; snrs=[]; l2_all=[]; l2_succ=[]; linf_all=[]; boundary_hits=[]
        for i in range(0, N, batch):
            Xi = X[i:i+batch].detach().clone().requires_grad_(True)
            yi = y[i:i+batch]
            xa = atk(Xi, yi).detach()

            # Optional LP variant
            if lp_sigma_t is not None:
                delta = xa - Xi
                delta = smooth_delta_gauss(delta, sigma_t=lp_sigma_t)
                delta = delta.clamp(-eps_z, eps_z)  # same L_inf budget in z-space
                xa = Xi + delta

            xa = per_channel_clamp(xa, train_min_t, train_max_t)

            with torch.no_grad():
                pa = model(xa).argmax(1)
            preds_adv.append(pa.cpu())

            d = (xa - Xi).flatten(1)
            l2v = d.norm(p=2, dim=1).detach().cpu().numpy()
            linfv = (xa - Xi).abs().flatten(1).max(dim=1).values.detach().cpu().numpy()
            l2_all.extend(l2v); linf_all.extend(linfv)

            succ = (pa != yi).cpu().numpy()
            l2_succ.extend(l2v[succ])

            snrs.extend(snr_db(Xi, xa))

            # boundary fraction (value-level, not sample-level)
            bmask = ((xa <= (train_min_t + 1e-12)) | (xa >= (train_max_t - 1e-12))).float()
            boundary_hits.append(float(bmask.mean().item()))

            # cache a consistent adversarial subset for explainers
            if len(adv_subset) < cap_N:
                take = min(cap_N - len(adv_subset), xa.size(0))
                adv_subset.append(xa[:take].detach())

        preds_adv = torch.cat(preds_adv)
        X_adv_cap = torch.cat(adv_subset, dim=0)
        y_cap     = y[cap_idx]

        # Explain on clean subset (cached) vs adv subset
        E_ADV = {name: fn(X_adv_cap, y_cap) for name, fn in EXPLAINERS.items()}

        adv_acc = float((preds_adv == y.cpu()).float().mean().item())

        # ---- common row (then add per-method metrics) ----
        row_core = {
            "attack": f"{atk_name}_LP" if lp_sigma_t is not None else atk_name,
            "smooth": bool(lp_sigma_t is not None),
            "smooth_type": "gaussian" if lp_sigma_t is not None else None,
            "smooth_sigma_t": float(lp_sigma_t) if lp_sigma_t is not None else None,
            "norm": "Linf",
            "muV_budget": float(mu_v),
            "eps_z": float(eps_z),
            "eps_uV_per_channel": eps_uV_per_channel(eps_z),
            "steps": kwargs.get("steps"),
            "alpha": float(kwargs["alpha"]) if "alpha" in kwargs else None,
            "random_start": bool(kwargs.get("random_start", False)),
            "seed": seed,
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
            "ASR": 1.0 - adv_acc,
            "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
            "mean_L2_all": float(np.mean(l2_all)) if len(l2_all) else float("nan"),
            "mean_Linf_delta": float(np.mean(linf_all)) if len(linf_all) else float("nan"),
            "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
            "clean_acc_per_class": clean_acc_pc,
            "adv_acc_per_class": per_class_acc(y.cpu(), preds_adv, num_classes),
            "frac_at_boundary": float(np.mean(boundary_hits)) if boundary_hits else float("nan"),
            "restarts": 1, "targeted": False
        }

        # ---- per-explainer metrics (IG only) ----
        for m in E_CLEAN.keys():
            Ec, Ea = E_CLEAN[m], E_ADV[m]
            # top-5 lists & ROI share of top-5
            top5_clean = topk_channels(Ec, k=5, ch_names=CH_NAMES)
            top5_adv   = topk_channels(Ea, k=5, ch_names=CH_NAMES)
            row_method = {
                f"spearman_{m}":             spearman_ch(Ec, Ea),
                f"roi_share_clean_{m}":      roi_share(Ec),
                f"roi_share_adv_{m}":        roi_share(Ea),
                f"roi_delta_share_{m}":      roi_share(Ea) - roi_share(Ec),
                f"roi_spearman_{m}":         roi_spearman(Ec, Ea),
                f"laterality_clean_{m}":     laterality_index(Ec),
                f"laterality_adv_{m}":       laterality_index(Ea),
                f"laterality_delta_{m}":     laterality_index(Ea) - laterality_index(Ec),
                f"top5_clean_{m}":           top5_clean,
                f"top5_adv_{m}":             top5_adv,
                f"top5_roi_share_clean_{m}": topk_roi_share(Ec, k=5),
                f"top5_roi_share_adv_{m}":   topk_roi_share(Ea, k=5),
                f"top5_roi_share_delta_{m}": topk_roi_share(Ea, k=5) - topk_roi_share(Ec, k=5),
            }
            rows.append({**row_core, **row_method})

    return rows

# ---------------- DeepFool (L2) ----------------
def run_deepfool(batch=128, seed=42, cap_idx=None, E_CLEAN=None):
    atk = ta.DeepFool(model, steps=50)
    preds_adv=[]; snrs=[]; l2_all=[]; l2_succ=[]; linf_all=[]; boundary_hits=[]
    N=X.size(0)

    # subset collector for explanations
    start = cap_idx.start or 0
    stop  = cap_idx.stop
    cap_N = stop - start
    adv_subset = []

    with torch.no_grad():
        preds_clean = model(X).argmax(1).cpu()
        clean_acc = float((preds_clean == y.cpu()).float().mean().item())

    for i in range(0, N, batch):
        Xi = X[i:i+batch].detach().clone().requires_grad_(True)
        yi = y[i:i+batch]
        xa = atk(Xi, yi).detach()
        xa = per_channel_clamp(xa, train_min_t, train_max_t)

        if len(adv_subset) < cap_N:
            take = min(cap_N - len(adv_subset), xa.size(0))
            adv_subset.append(xa[:take].detach())

        with torch.no_grad():
            pa = model(xa).argmax(1)
        preds_adv.append(pa.cpu())

        d = (xa - Xi).flatten(1)
        l2v = d.norm(p=2, dim=1).detach().cpu().numpy()
        l2_all.extend(l2v)
        linfv = (xa - Xi).abs().flatten(1).max(dim=1).values.detach().cpu().numpy()
        linf_all.extend(linfv)

        succ = (pa != yi).cpu().numpy()
        l2_succ.extend(l2v[succ])

        snrs.extend(snr_db(Xi, xa))
        bmask = ((xa <= (train_min_t + 1e-12)) | (xa >= (train_max_t - 1e-12))).float()
        boundary_hits.append(float(bmask.mean().item()))

    preds_adv = torch.cat(preds_adv)
    adv_acc = float((preds_adv == y.cpu()).float().mean().item())

    X_adv_cap = torch.cat(adv_subset, dim=0)
    y_cap     = y[cap_idx]
    E_ADV = {name: fn(X_adv_cap, y_cap) for name, fn in EXPLAINERS.items()}

    rows=[]
    for m in E_CLEAN.keys():
        Ec, Ea = E_CLEAN[m], E_ADV[m]
        top5_clean = topk_channels(Ec, k=5, ch_names=CH_NAMES)
        top5_adv   = topk_channels(Ea, k=5, ch_names=CH_NAMES)
        rows.append({
            "attack": "DeepFool_L2",
            "norm": "L2",
            "muV_budget": None,
            "eps_z": None,
            "eps_uV_per_channel": None,
            "steps": 50, "alpha": None, "random_start": None,
            "seed": seed,
            "clean_acc": clean_acc, "adv_acc": adv_acc, "ASR": 1.0 - adv_acc,
            "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
            "mean_L2_all": float(np.mean(l2_all)) if len(l2_all) else float("nan"),
            "mean_Linf_delta": float(np.mean(linf_all)) if len(linf_all) else float("nan"),
            "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
            "frac_at_boundary": float(np.mean(boundary_hits)) if boundary_hits else float("nan"),
            "restarts": 1, "targeted": False,
            f"spearman_{m}":             spearman_ch(Ec, Ea),
            f"roi_share_clean_{m}":      roi_share(Ec),
            f"roi_share_adv_{m}":        roi_share(Ea),
            f"roi_delta_share_{m}":      roi_share(Ea) - roi_share(Ec),
            f"roi_spearman_{m}":         roi_spearman(Ec, Ea),
            f"laterality_clean_{m}":     laterality_index(Ec),
            f"laterality_adv_{m}":       laterality_index(Ea),
            f"laterality_delta_{m}":     laterality_index(Ea) - laterality_index(Ec),
            f"top5_clean_{m}":           top5_clean,
            f"top5_adv_{m}":             top5_adv,
            f"top5_roi_share_clean_{m}": topk_roi_share(Ec, k=5),
            f"top5_roi_share_adv_{m}":   topk_roi_share(Ea, k=5),
            f"top5_roi_share_delta_{m}": topk_roi_share(Ea, k=5) - topk_roi_share(Ec, k=5),
        })
    return rows


In [105]:
# ========= Cell 3: Subject setup + full run =========
import os, json, torch, pandas as pd
# Assumes your model classes and `load_adv_test_data(subject_id)` are already defined/imported:
#   EEGNetv4, Deep4Net, CTNet, EEGMamba, load_adv_test_data

# ---- seeds (report uses multiple) ----
SEEDS = [42, 123, 2024, 31415, 999]

# ---- model builders and checkpoints ----
MODEL_BUILDERS = {
    "EEGNet":      lambda: EEGNetv4(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "DeepConvNet": lambda: Deep4Net(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "CTNet":       lambda: CTNet(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "Mamba":       lambda: EEGMamba(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
}

CKPTS = {
    "EEGNet":       {s: f"results/EEGNet/EEGNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"          for s in SEEDS},
    "DeepConvNet":  {s: f"results/DeepConvNet/DeepConvNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth" for s in SEEDS},
    "CTNet":        {s: f"results/CTNet/CTNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"             for s in SEEDS},
    "Mamba":        {s: f"results/EEGMamba/EEGMamba_S{SUBJECT_ID}_seed{s}/checkpoint.pth"       for s in SEEDS},
}

# ---- device ----
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- subject setup (loads data + stats; robust to 6- or 7-tuple loaders) ----
def setup_subject(subject_id):
    global X, y, train_min_t, train_max_t, train_std_np, CH_NAMES
    global prenorm_std_np, median_std_pre
    global ATTR_MAX_N, N_STEPS_IG, IG_INT_BS, ROI_IDX

    data = load_adv_test_data(subject_id)
    if isinstance(data, (list, tuple)) and len(data) >= 7:
        X, y, train_min_t, train_max_t, train_std_np, CH_NAMES, prenorm_std_np = data[:7]
    else:
        X, y, train_min_t, train_max_t, train_std_np, CH_NAMES = data
        # Fallback if prenorm stds not provided by loader (use EMS stds as proxy)
        prenorm_std_np = train_std_np

    # map μV -> εᶻ uses PRENORM stats
    median_std_pre = float(np.median(prenorm_std_np))

    # ROI indices for this subject’s channels
    ROI_IDX = [CH_NAMES.index(n) for n in ("C3","C4","Cz") if n in CH_NAMES]

    # IG attribution settings
    ATTR_MAX_N = 128
    N_STEPS_IG = 16
    IG_INT_BS  = 8  # not used now (we pass internal_bs=Xi.size(0))

    # move data to device
    X = X.to(device); y = y.to(device)
    train_min_t = train_min_t.to(device); train_max_t = train_max_t.to(device)

def set_all_seeds(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def run_for_model_seed(model_name: str, seed: int):
    global model, ig, ATTR_MAX_N, N_STEPS_IG

    set_all_seeds(seed)

    # build & load
    m = MODEL_BUILDERS[model_name]().to(device)
    ckpt_path = CKPTS[model_name][seed]
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    m.load_state_dict(state_dict)
    m.eval()
    model = m

    # IG explainer
    ig = CA.IntegratedGradients(model)

    # fixed clean subset for attributions
    IDX_CAP = slice(0, min(ATTR_MAX_N, X.size(0)))
    X_cap_clean = X[IDX_CAP].detach().clone()
    y_cap       = y[IDX_CAP].detach().clone()

    # clean explanations cache
    E_CLEAN = {name: fn(X_cap_clean, y_cap) for name, fn in EXPLAINERS.items()}

    rows = []
    # FGSM / PGD standard
    rows += run_linf_sweep("FGSM", muV_grid, steps=None, lp_sigma_t=None, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)
    rows += run_linf_sweep("PGD",  muV_grid, steps=40,    alpha_rule=lambda e: e/8, lp_sigma_t=None, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # low-pass (LP) variants (σᵗ=3.0)
    rows += run_linf_sweep("FGSM", muV_grid, steps=None, lp_sigma_t=3.0, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)
    rows += run_linf_sweep("PGD",  muV_grid, steps=40,    alpha_rule=lambda e: e/8, lp_sigma_t=3.0, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # DeepFool (L2)
    rows += run_deepfool(cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # tag rows
    for r in rows:
        r["subject_id"] = SUBJECT_ID
        r["model_name"] = model_name
        r["seed"] = seed
    return rows

# ===== Run per-subject across models × seeds =====
SUBJECT_ID = 1
setup_subject(SUBJECT_ID)

n_channels = int(X.shape[1])
n_times    = int(X.shape[2])
n_classes  = int(y.max().item() + 1)
print(f"Data shape: {X.shape} | n_ch={n_channels}, n_times={n_times}, n_classes={n_classes}")
print(f"Median PRENORM std (μV): {median_std_pre:.6f} | μV grid: {muV_grid}")

all_rows = []
for model_name in MODEL_BUILDERS.keys():
    print(f"Running: {model_name}")
    for seed in SEEDS:
        all_rows.extend(run_for_model_seed(model_name, seed))

# ---- save per-subject CSV ----
os.makedirs("results", exist_ok=True)
csv_path = f"results/adversarial_results_{SUBJECT_ID}.csv"
pd.DataFrame(all_rows).to_csv(csv_path, index=False)
print(f"Wrote {csv_path} with {len(all_rows)} rows.")

# ---- update master ----
master_path = "results/adversarial_results_MASTER.csv"
if os.path.isfile(master_path):
    pd.concat([pd.read_csv(master_path), pd.DataFrame(all_rows)], ignore_index=True).to_csv(master_path, index=False)
else:
    pd.DataFrame(all_rows).to_csv(master_path, index=False)
print(f"Updated {master_path}.")


Data shape: torch.Size([288, 22, 1125]) | n_ch=22, n_times=1125, n_classes=4
Median PRENORM std (μV): 6.870272 | μV grid: [0.25, 0.5, 1.0, 2.0]
Running: EEGNet
Running: DeepConvNet
Running: CTNet
Running: Mamba
Wrote results/adversarial_results_1.csv with 340 rows.
Updated results/adversarial_results_MASTER.csv.


## White‑box L∞ attacks (FGSM, BIM, PGD, MIM)

In [106]:
!git add .
!git commit -m 'attacks s1'
! git push origin main

[main 90f2e4b] attacks s1
 3 files changed, 682 insertions(+), 682 deletions(-)
 create mode 100644 results/S1_prenorm_stats.npz
 rewrite results/adversarial_results_1.csv (97%)
 rewrite results/adversarial_results_MASTER.csv (97%)
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 31.95 KiB | 4.56 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/VictoryChianumba/robust-eeg-models.git
   263a434..90f2e4b  main -> main


## L₂‑style attacks (DeepFool, CW)

## Masking sanity check (AutoAttack subset) + black‑box (Square, FAB)

# Save & pretty‑print

In [66]:
# --- Sanity test: DeepFool (L2) + IG stability & ROI metrics, per model (seed=42) ---
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F
from captum.attr import IntegratedGradients
import torchattacks as ta
from scipy.stats import kendalltau

# --------- tiny helpers (self-contained) ----------


def _per_trial_stats(vals):
    vals = np.array([v for v in vals if np.isfinite(v)])
    if vals.size == 0:
        return float("nan"), float("nan")
    med = float(np.median(vals))
    iqr = float(np.percentile(vals, 75) - np.percentile(vals, 25))
    return med, iqr

def set_all_seeds(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def per_channel_clamp(x, vmin, vmax):
    return torch.max(torch.min(x, vmax), vmin)

def snr_db(x, x_adv):
    d = x_adv - x
    num = x.pow(2).sum((1,2)).sqrt()
    den = d.pow(2).sum((1,2)).sqrt().clamp_min(1e-12)
    return (20.0 * torch.log10(num/den)).detach().cpu().numpy()

def channel_scores(E):  # (N,C,T) -> (N,C)
    return E.abs().mean(dim=-1).detach().cpu().numpy()

from scipy.stats import spearmanr
def spearman_ch(Ec, Ea):
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i], sa[i]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

# ROI support (uses global CH_NAMES / ROI_IDX if present; otherwise safe fallbacks)
try:
    CHN = CH_NAMES[:]  # from your main setup (if available)
except Exception:
    CHN = [f"ch{i}" for i in range(int(X.shape[1]))]

try:
    ROI_IDX = ROI_IDX[:]  # from your main setup (if available)
except Exception:
    # Default BNCI2014-001 motor ROI indices if names exist, else None
    try:
        roi_names = ["C3","C4","Cz"]
        ROI_IDX = [CHN.index(n) for n in roi_names if n in CHN]
        if len(ROI_IDX) == 0: ROI_IDX = None
    except Exception:
        ROI_IDX = None

def roi_share(E, roi_idx=None):
    if roi_idx is None: roi_idx = ROI_IDX
    if roi_idx is None: return float("nan")
    sc = channel_scores(E).mean(axis=0)  # (C,)
    total = sc.sum() + 1e-12
    return float(sc[roi_idx].sum() / total)

def roi_spearman(Ec, Ea, roi_idx=None):
    if roi_idx is None: roi_idx = ROI_IDX
    if roi_idx is None: return float("nan")
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i, roi_idx], sa[i, roi_idx]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

def laterality_index(E, left="C3", right="C4"):
    # Requires CHN having those names
    if left not in CHN or right not in CHN: return float("nan")
    li, ri = CHN.index(left), CHN.index(right)
    sc = channel_scores(E).mean(axis=0)
    num = sc[li] - sc[ri]
    den = sc[li] + sc[ri] + 1e-8
    return float(num/den)

def attr_IG(model, X_in, y_in, n_steps=16, internal_bs=8, max_n=None):
    if max_n is None: max_n = internal_bs
    if X_in.size(0) > max_n:
        Xs, ys = X_in[:max_n], y_in[:max_n]
    else:
        Xs, ys = X_in, y_in
    ig = IntegratedGradients(model)
    return ig.attribute(
        Xs.detach().clone().requires_grad_(True),
        target=ys, n_steps=n_steps,
        baselines=torch.zeros_like(Xs),
        internal_batch_size=internal_bs
    )

# --------- assumptions / fallbacks from your main notebook ----------
device = "cuda" if torch.cuda.is_available() else "cpu"

# Expect global: MODEL_BUILDERS, CKPTS, X, y, train_min_t, train_max_t
# Provide safe fallbacks for vmin/vmax if not present (uses test set bounds)
try:
    VMN, VMX = train_min_t, train_max_t
except Exception:
    vmn = X.min(dim=2, keepdim=True).values.min(dim=0, keepdim=True).values
    vmx = X.max(dim=2, keepdim=True).values.max(dim=0, keepdim=True).values
    VMN, VMX = vmn.to(device), vmx.to(device)

# --------- config for this probe ----------
SEED = 42
BATCH = 128
ATTR_MAX_N = 8        # keep = internal_bs to avoid Captum override warning
IG_STEPS   = 16
IG_INT_BS  = 8
IDX_CAP    = slice(0, min(ATTR_MAX_N, X.size(0)))   # attribution subset
IDX_ATTACK = slice(0, X.size(0))                    # run DeepFool on full test set (change to smaller slice if needed)

# Models to test (pull from your builders)
model_names = list(MODEL_BUILDERS.keys())

# --------- run per model ----------
set_all_seeds(SEED)
IG_CLEAN_MEANS = {}
rows = []

for model_name in model_names:
    print(f"\n=== Testing {model_name} (DeepFool + IG sanity) ===")
    try:
        # 1) build and load
        m = MODEL_BUILDERS[model_name]().to(device)
        ckpt_path = CKPTS[model_name][SEED]
        if not os.path.isfile(ckpt_path):
            print(f"[WARN] Missing ckpt: {ckpt_path}. Skipping.")
            continue
        sd = torch.load(ckpt_path, map_location=device)
        state_dict = sd["state_dict"] if "state_dict" in sd else sd
        if any(k.startswith("module.") for k in state_dict.keys()):
            state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
        m.load_state_dict(state_dict); m.eval()

        # 2) clean preds (on full slice for accuracy)
        with torch.no_grad():
            preds_clean = m(X[IDX_ATTACK]).argmax(1).cpu()
        clean_acc = float((preds_clean == y[IDX_ATTACK].cpu()).float().mean().item())

        # 3) clean IG (on capped subset), flatness probe
        X_cap = X[IDX_CAP]; y_cap = y[IDX_CAP]
        E_clean = attr_IG(m, X_cap, y_cap, n_steps=IG_STEPS, internal_bs=IG_INT_BS, max_n=ATTR_MAX_N)
        cs = channel_scores(E_clean)             # (N_cap, C)
        flat_var_mean = float(cs.var(axis=1).mean())
        flat_var_med  = float(np.median(cs.var(axis=1)))
        top5_idx = np.argsort(-cs.mean(axis=0))[:5]
        top5_names = [CHN[i] for i in top5_idx]
        print(f"[IG sanity] mean var(ch)={flat_var_mean:.3e} | median={flat_var_med:.3e} | top-5 {top5_names}")

        # cache normalized mean clean IG map for cross-model cosine
        v = E_clean.abs().mean(dim=0).flatten().detach().cpu().numpy()
        v = v / (np.linalg.norm(v) + 1e-12)
        IG_CLEAN_MEANS[(model_name, SEED)] = v

        # 4) DeepFool on attack slice
        atk = ta.DeepFool(m, steps=50)
        preds_adv=[]; snrs=[]; l2_all=[]; l2_succ=[]
        N = X[IDX_ATTACK].size(0)
        for i in range(0, N, BATCH):
            Xi = X[IDX_ATTACK][i:i+BATCH].detach().clone().requires_grad_(True)
            yi = y[IDX_ATTACK][i:i+BATCH]
            xa = atk(Xi, yi).detach()
            xa = per_channel_clamp(xa, VMN, VMX)
            with torch.no_grad():
                pa = m(xa).argmax(1)
            preds_adv.append(pa.cpu())
            d = (xa - Xi).flatten(1); l2v = d.norm(p=2, dim=1).detach().cpu().numpy()
            l2_all.extend(l2v)
            succ = (pa != yi).cpu().numpy()
            l2_succ.extend(l2v[succ]); snrs.extend(snr_db(Xi, xa))
        preds_adv = torch.cat(preds_adv)
        adv_acc = float((preds_adv == y[IDX_ATTACK].cpu()).float().mean().item())

        # 5) ADV IG on same capped subset + metrics
        # build adv subset aligned to cap (re-run DeepFool only for cap to avoid storing all)
        with torch.no_grad():
            # small re-run for cap to get X_adv_cap
            Xi = X_cap.detach().clone().requires_grad_(True)
        X_adv_cap = ta.DeepFool(m, steps=50)(Xi, y_cap).detach()
        X_adv_cap = per_channel_clamp(X_adv_cap, VMN, VMX)
        E_adv = attr_IG(m, X_adv_cap, y_cap, n_steps=IG_STEPS, internal_bs=IG_INT_BS, max_n=ATTR_MAX_N)

        row = {
            "model": model_name,
            "seed": SEED,
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
            "ASR": 1.0 - adv_acc,
            "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
            "mean_L2_all": float(np.mean(l2_all)) if len(l2_all) else float("nan"),
            "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
            # IG stability & ROI
            "spearman_IG": spearman_ch(E_clean, E_adv),
            "roi_share_clean_IG": roi_share(E_clean),
            "roi_share_adv_IG":   roi_share(E_adv),
            "roi_delta_share_IG": (roi_share(E_adv) - roi_share(E_clean))
                                   if np.isfinite(roi_share(E_clean)) and np.isfinite(roi_share(E_adv)) else float("nan"),
            "roi_spearman_IG":    roi_spearman(E_clean, E_adv),
            "laterality_clean_IG": laterality_index(E_clean),
            "laterality_adv_IG":   laterality_index(E_adv),
        }
        row["laterality_delta_IG"] = (row["laterality_adv_IG"] - row["laterality_clean_IG"]
                                      if np.isfinite(row["laterality_clean_IG"]) and np.isfinite(row["laterality_adv_IG"])
                                      else float("nan"))
        rows.append(row)

        print(pd.Series(row)[["clean_acc","adv_acc","ASR","spearman_IG",
                              "roi_share_clean_IG","roi_share_adv_IG","roi_spearman_IG"]])

    except Exception as e:
        print(f"{model_name}: FAILED -> {e}")

# --------- cross-model cosine (clean IG maps) ----------
seed_pick = SEED
keys = [(m, s) for (m, s) in IG_CLEAN_MEANS.keys() if s == seed_pick]
if keys:
    mats = np.stack([IG_CLEAN_MEANS[k] for k in keys], axis=0)
    cos = mats @ mats.T
    labels = [k[0] for k in keys]
    print("\nCosine similarity (clean IG mean maps)")
    print(pd.DataFrame(cos, index=labels, columns=labels))
else:
    print(f"[sanity] No cached IG maps for seed {seed_pick}.")

# Final compact table
print("\nSummary:")
print(pd.DataFrame(rows)[["model","clean_acc","adv_acc","ASR","spearman_IG",
                          "roi_share_clean_IG","roi_share_adv_IG","roi_spearman_IG"]])



=== Testing EEGNet (DeepFool + IG sanity) ===
[IG sanity] mean var(ch)=1.742e-05 | median=1.824e-05 | top-5 ['P1', 'POz', 'FC3', 'C2', 'CP4']
clean_acc             0.663194
adv_acc               0.100694
ASR                   0.899306
spearman_IG           0.957933
roi_share_clean_IG    0.123006
roi_share_adv_IG      0.122039
roi_spearman_IG         0.9375
dtype: object

=== Testing DeepConvNet (DeepFool + IG sanity) ===
[IG sanity] mean var(ch)=4.446e-06 | median=4.049e-06 | top-5 ['CP3', 'C1', 'CP4', 'Cz', 'C5']
clean_acc             0.586806
adv_acc               0.100694
ASR                   0.899306
spearman_IG           0.971767
roi_share_clean_IG    0.161112
roi_share_adv_IG      0.160864
roi_spearman_IG            0.5
dtype: object

=== Testing CTNet (DeepFool + IG sanity) ===
[IG sanity] mean var(ch)=1.498e-05 | median=1.382e-05 | top-5 ['P1', 'C1', 'CP3', 'CP1', 'POz']
clean_acc             0.694444
adv_acc                0.09375
ASR                    0.90625
spearman_IG  

## For TSNE and PCA illustrations

In [101]:
# ==== FGSM/PGD sanity checker (fixed capping; EMS space) ====
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import spearmanr, kendalltau
import captum.attr as CA
import torchattacks as ta

# ---------- helpers ----------
def channel_scores(E):  # (N,C,T) -> (N,C)
    return E.abs().mean(dim=-1).detach().cpu().numpy()

def spearman_ch(Ec, Ea):
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i], sa[i]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

def kendallb_mean(Ec, Ea):
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = kendalltau(sc[i], sa[i], variant="b").correlation
        if r is not None and not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

def jaccard_topk(Ec, Ea, k=5):
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        kc = np.argsort(-sc[i])[:k]; ka = np.argsort(-sa[i])[:k]
        inter = len(set(kc).intersection(set(ka)))
        union = len(set(kc).union(set(ka)))
        vals.append(inter/union if union>0 else np.nan)
    vals = [v for v in vals if not np.isnan(v)]
    return float(np.mean(vals)) if vals else float("nan")

def cosine_maps(Ec, Ea):
    v1 = Ec.flatten(1).detach().cpu().numpy()
    v2 = Ea.flatten(1).detach().cpu().numpy()
    v1n = v1 / (np.linalg.norm(v1, axis=1, keepdims=True)+1e-12)
    v2n = v2 / (np.linalg.norm(v2, axis=1, keepdims=True)+1e-12)
    return float(np.mean(np.sum(v1n*v2n, axis=1)))

# ROI helpers
try:
    CH_NAMES
except NameError:
    CH_NAMES = None
ROI_NAMES = ("C3","C4","Cz")
ROI_IDX = [CH_NAMES.index(n) for n in ROI_NAMES if (CH_NAMES is not None and n in CH_NAMES)]

def roi_share(E):
    if not ROI_IDX: return float("nan")
    sc = channel_scores(E).mean(axis=0)
    return float(sc[ROI_IDX].sum() / (sc.sum() + 1e-12))

def roi_spearman(Ec, Ea):
    if not ROI_IDX: return float("nan")
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        r = spearmanr(sc[i, ROI_IDX], sa[i, ROI_IDX]).statistic
        if not np.isnan(r): vals.append(r)
    return float(np.mean(vals)) if vals else float("nan")

def roi_topk_share(E, k=5):
    if not ROI_IDX: return float("nan")
    sc = channel_scores(E)
    vals=[]
    for i in range(sc.shape[0]):
        topk = np.argsort(-sc[i])[:k]
        vals.append(len(set(topk).intersection(ROI_IDX))/k)
    return float(np.mean(vals))

def roi_topk_jaccard(Ec, Ea, k=5):
    if not ROI_IDX: return float("nan")
    sc, sa = channel_scores(Ec), channel_scores(Ea)
    vals=[]
    for i in range(sc.shape[0]):
        topk_c = set(np.argsort(-sc[i])[:k]).intersection(ROI_IDX)
        topk_a = set(np.argsort(-sa[i])[:k]).intersection(ROI_IDX)
        inter = len(topk_c & topk_a); union = len(topk_c | topk_a)
        vals.append(inter/union if union>0 else 1.0)
    return float(np.mean(vals))

def per_class_acc(y_true, y_pred, nclass=None):
    yt = y_true.detach().cpu().numpy(); yp = y_pred.detach().cpu().numpy()
    K = int(nclass) if nclass is not None else int(max(yp.max(), yt.max()) + 1)
    out=[]
    for c in range(K):
        idx = (yt == c)
        out.append(float((yp[idx] == c).mean()) if idx.any() else float("nan"))
    return out

def smooth_delta_gauss(delta_t: torch.Tensor, sigma_t: float) -> torch.Tensor:
    if sigma_t is None or sigma_t <= 0: return delta_t
    device = delta_t.device
    radius = int(4 * sigma_t + 0.5)
    x = torch.arange(-radius, radius + 1, dtype=delta_t.dtype, device=device)
    kernel = torch.exp(-0.5 * (x / sigma_t).pow(2))
    kernel /= kernel.sum()
    C = delta_t.size(1)
    kernel = kernel.expand(C, 1, -1)
    return F.conv1d(delta_t, kernel, padding=radius, groups=C)

# ---------- core per-model runner ----------
def run_model_fgsm_pgd_sanity(model_name, seed=42,
                              eps_grid=(0.01,0.02,0.04,0.08),
                              lp_sigma_t=3.0,
                              attr_max_n=128,
                              ig_steps=16,
                              ig_int_bs=8,
                              batch=128,
                              subset_cal=512,
                              target_asr=0.5,
                              bisection_steps=3):
    # build + load model
    m = MODEL_BUILDERS[model_name]().to(device)
    ckpt_path = CKPTS[model_name][seed]
    sd = torch.load(ckpt_path, map_location=device)
    sd = sd["state_dict"] if "state_dict" in sd else sd
    if any(k.startswith("module.") for k in sd.keys()):
        sd = {k.replace("module.","",1): v for k,v in sd.items()}
    m.load_state_dict(sd); m.eval()

    ig = CA.IntegratedGradients(m)

    # clean preds + clean IG on a fixed subset
    with torch.no_grad():
        num_classes = int(m(X[:1]).shape[-1])
        preds_clean = m(X).argmax(1).cpu()
        clean_acc = float((preds_clean == y.cpu()).float().mean().item())

    cap_N = min(attr_max_n, X.size(0))
    Xc = X[:cap_N].detach().clone()
    yc = y[:cap_N].detach().clone()
    E_CLEAN = ig.attribute(
        Xc, target=yc, n_steps=ig_steps,
        baselines=torch.zeros_like(Xc),
        internal_batch_size=min(ig_int_bs, Xc.size(0))
    )

    # IG sanity print
    sc = channel_scores(E_CLEAN).var(axis=0)
    top5 = np.array(CH_NAMES)[np.argsort(-sc)[:5]] if CH_NAMES else np.argsort(-sc)[:5]
    print(f"[{model_name} | IG sanity] mean var(ch)={sc.mean():.3e} | top-5 {list(top5)}")

    def eval_attack(atk_name, eps, use_lp):
        ctor = {"FGSM": ta.FGSM, "PGD": ta.PGD}[atk_name]
        kwargs = {}
        if atk_name == "PGD":
            kwargs.update(dict(steps=40, alpha=float(eps)/10.0, random_start=True))
        atk = ctor(m, eps=float(eps), **kwargs)

        preds_adv=[]; l2_all=[]; linf_all=[]; at_boundary=[]
        adv_cap_chunks = []
        collected = 0  # number of ADV examples stored for attribution

        N = X.size(0)
        for i in range(0, N, batch):
            Xi = X[i:i+batch].detach().clone().requires_grad_(True)
            yi = y[i:i+batch]
            xa = atk(Xi, yi).detach()
            delta = (xa - Xi).clamp(-eps, eps)
            xa = Xi + delta
            if use_lp:
                delta = xa - Xi
                delta = smooth_delta_gauss(delta, lp_sigma_t)
                delta = delta.clamp(-eps, eps)
                xa = Xi + delta

            with torch.no_grad():
                pa = m(xa).argmax(1)
            preds_adv.append(pa.cpu())

            delta = (xa - Xi)
            l2v   = delta.flatten(1).norm(p=2, dim=1).detach().cpu().numpy()
            linfv = delta.abs().flatten(1).amax(dim=1).detach().cpu().numpy()
            l2_all.extend(l2v); linf_all.extend(linfv)
            at_boundary.extend((linfv >= (0.99*eps)).astype(np.float32).tolist())

            # cap ADV examples to match yc size exactly
            if collected < cap_N:
                take = min(cap_N - collected, xa.size(0))
                adv_cap_chunks.append(xa[:take].detach())
                collected += take

        preds_adv = torch.cat(preds_adv)
        adv_acc = float((preds_adv == y.cpu()).float().mean().item())

        # build capped ADV tensor with exact size = cap_N
        X_adv_cap = torch.cat(adv_cap_chunks, dim=0)
        if X_adv_cap.size(0) > cap_N:
            X_adv_cap = X_adv_cap[:cap_N]
        elif X_adv_cap.size(0) < cap_N:
            # should not happen, but guard against it by repeating last few
            pad = cap_N - X_adv_cap.size(0)
            X_adv_cap = torch.cat([X_adv_cap, X_adv_cap[-1:].repeat(pad,1,1)], dim=0)

        E_ADV = ig.attribute(
            X_adv_cap, target=yc, n_steps=ig_steps,
            baselines=torch.zeros_like(X_adv_cap),
            internal_batch_size=min(ig_int_bs, X_adv_cap.size(0))
        )

        metrics = dict(
            spearman_IG = spearman_ch(E_CLEAN, E_ADV),
            kendallb_mean_IG = kendallb_mean(E_CLEAN, E_ADV),
            jaccard_top5_IG = jaccard_topk(E_CLEAN, E_ADV, k=5),
            cosine_maps_IG = cosine_maps(E_CLEAN, E_ADV),
            roi_share_clean_IG = roi_share(E_CLEAN),
            roi_share_adv_IG   = roi_share(E_ADV),
            roi_spearman_IG    = roi_spearman(E_CLEAN, E_ADV),
            roi_topk_share_clean_IG = roi_topk_share(E_CLEAN, k=5),
            roi_topk_share_adv_IG   = roi_topk_share(E_ADV,   k=5),
            roi_topk_share_delta_IG = (roi_topk_share(E_ADV, k=5) - roi_topk_share(E_CLEAN, k=5)),
        )

        row = dict(
            model=model_name,
            attack=f"{atk_name}_LP" if use_lp else atk_name,
            eps_z=float(eps),
            smooth=bool(use_lp),
            clean_acc=clean_acc,
            adv_acc=adv_acc,
            ASR=1.0 - adv_acc,
            mean_L2_all=float(np.mean(l2_all)) if l2_all else float("nan"),
            mean_Linf_delta=float(np.mean(linf_all)) if linf_all else float("nan"),
            frac_at_boundary=float(np.mean(at_boundary)) if at_boundary else float("nan"),
        )
        row.update(metrics)
        return row

    # sweep (vanilla + LP)
    rows=[]
    for eps in eps_grid:
        rows.append(eval_attack("FGSM", eps, use_lp=False))
        rows.append(eval_attack("FGSM", eps, use_lp=True))
    for eps in eps_grid:
        rows.append(eval_attack("PGD", eps, use_lp=False))
        rows.append(eval_attack("PGD", eps, use_lp=True))

    # matched-ε snapshot (PGD ASR ~ 0.5 on a subset)
    def asr_on_subset(eps):
        atk = ta.PGD(m, eps=float(eps), steps=40, alpha=float(eps)/10.0, random_start=True)
        idx = slice(0, min(subset_cal, X.size(0)))
        preds=[]
        for i in range(idx.start, idx.stop, batch):
            Xi = X[i:i+batch].detach().clone().requires_grad_(True)
            yi = y[i:i+batch]
            xa = atk(Xi, yi).detach()
            with torch.no_grad():
                preds.append(m(xa).argmax(1).cpu())
        preds = torch.cat(preds)
        adv_acc = float((preds == y[idx].cpu()).float().mean().item())
        return 1.0 - adv_acc

    lo, hi = 0.005, 0.08
    for _ in range(bisection_steps):
        mid = (lo+hi)/2.0
        asr = asr_on_subset(mid)
        if asr < target_asr: lo = mid
        else: hi = mid
    eps_star = (lo+hi)/2.0
    rows.append(eval_attack("PGD", eps_star, use_lp=False))
    rows.append(eval_attack("PGD", eps_star, use_lp=True))

    return pd.DataFrame(rows)

# ---- run across models ----
ALL = []
for model_name in ["EEGNet","DeepConvNet","CTNet","Mamba"]:
    print(f"\n=== FGSM/PGD sanity: {model_name} ===")
    df = run_model_fgsm_pgd_sanity(model_name)
    view_cols = ["attack","eps_z","smooth","adv_acc","ASR","spearman_IG","kendallb_mean_IG","jaccard_top5_IG","cosine_maps_IG",
                 "roi_share_clean_IG","roi_share_adv_IG","roi_topk_share_clean_IG","roi_topk_share_adv_IG","roi_topk_share_delta_IG",
                 "mean_Linf_delta","frac_at_boundary"]
    print(df[view_cols].to_string(index=False))
    ALL.append(df)

SUM = pd.concat(ALL, ignore_index=True)
print("\nSummary (all models):")
print(SUM[["model","attack","eps_z","smooth","adv_acc","ASR","spearman_IG","kendallb_mean_IG","jaccard_top5_IG","cosine_maps_IG"]].to_string(index=False))



=== FGSM/PGD sanity: EEGNet ===


/usr/local/lib/python3.12/dist-packages/captum/attr/_utils/batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 128 equal to the number of examples.
  warnings.warn(


[EEGNet | IG sanity] mean var(ch)=6.134e-06 | top-5 ['P1', 'POz', 'CP4', 'Cz', 'CP3']
 attack    eps_z  smooth  adv_acc      ASR  spearman_IG  kendallb_mean_IG  jaccard_top5_IG  cosine_maps_IG  roi_share_clean_IG  roi_share_adv_IG  roi_topk_share_clean_IG  roi_topk_share_adv_IG  roi_topk_share_delta_IG  mean_Linf_delta  frac_at_boundary
   FGSM 0.010000   False 0.468750 0.531250     0.997150          0.981399         0.976562        0.999102             0.12501          0.125207                 0.079687               0.082812                 0.003125         0.010000               1.0
FGSM_LP 0.010000    True 0.489583 0.510417     0.997333          0.982396         0.979167        0.999168             0.12501          0.125215                 0.079687               0.082812                 0.003125         0.010000               1.0
   FGSM 0.020000   False 0.239583 0.760417     0.993409          0.961986         0.947917        0.996531             0.12501          0.125355           

/usr/local/lib/python3.12/dist-packages/captum/attr/_utils/batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 128 equal to the number of examples.
  warnings.warn(


[DeepConvNet | IG sanity] mean var(ch)=1.125e-06 | top-5 ['CP3', 'C1', 'Cz', 'C3', 'C5']
 attack    eps_z  smooth  adv_acc      ASR  spearman_IG  kendallb_mean_IG  jaccard_top5_IG  cosine_maps_IG  roi_share_clean_IG  roi_share_adv_IG  roi_topk_share_clean_IG  roi_topk_share_adv_IG  roi_topk_share_delta_IG  mean_Linf_delta  frac_at_boundary
   FGSM 0.010000   False 0.541667 0.458333     0.998580          0.990260         0.979167        0.997249            0.164032          0.164046                 0.271875               0.273438                 0.001562         0.010000               1.0
FGSM_LP 0.010000    True 0.576389 0.423611     0.999153          0.994048         0.992188        0.999042            0.164032          0.164032                 0.271875               0.273438                 0.001562         0.010000               1.0
   FGSM 0.020000   False 0.475694 0.524306     0.997168          0.982075         0.958333        0.993495            0.164032          0.164049        

/usr/local/lib/python3.12/dist-packages/captum/attr/_utils/batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 128 equal to the number of examples.
  warnings.warn(


[CTNet | IG sanity] mean var(ch)=3.850e-06 | top-5 ['P1', 'FC2', 'CP3', 'Cz', 'CP2']
 attack    eps_z  smooth  adv_acc      ASR  spearman_IG  kendallb_mean_IG  jaccard_top5_IG  cosine_maps_IG  roi_share_clean_IG  roi_share_adv_IG  roi_topk_share_clean_IG  roi_topk_share_adv_IG  roi_topk_share_delta_IG  mean_Linf_delta  frac_at_boundary
   FGSM 0.010000   False 0.520833 0.479167     0.998924          0.992289         0.981771        0.999757            0.126473          0.126423                 0.082812               0.084375             1.562500e-03         0.010000               1.0
FGSM_LP 0.010000    True 0.538194 0.461806     0.999021          0.993033         0.981771        0.999815            0.126473          0.126417                 0.082812               0.084375             1.562500e-03         0.010000               1.0
   FGSM 0.020000   False 0.329861 0.670139     0.997662          0.984443         0.968750        0.999031            0.126473          0.126380            

/usr/local/lib/python3.12/dist-packages/captum/attr/_utils/batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 128 equal to the number of examples.
  warnings.warn(


[Mamba | IG sanity] mean var(ch)=2.602e-04 | top-5 ['CP3', 'Cz', 'C3', 'C5', 'FC3']
 attack    eps_z  smooth  adv_acc      ASR  spearman_IG  kendallb_mean_IG  jaccard_top5_IG  cosine_maps_IG  roi_share_clean_IG  roi_share_adv_IG  roi_topk_share_clean_IG  roi_topk_share_adv_IG  roi_topk_share_delta_IG  mean_Linf_delta  frac_at_boundary
   FGSM 0.010000   False 0.565972 0.434028     0.995897          0.974297         0.960938        0.993022            0.182305          0.182680                 0.365625               0.364063                -0.001563         0.010000               1.0
FGSM_LP 0.010000    True 0.614583 0.385417     0.997291          0.982008         0.976562        0.996749            0.182305          0.182558                 0.365625               0.364063                -0.001563         0.010000               1.0
   FGSM 0.020000   False 0.364583 0.635417     0.991124          0.952922         0.937500        0.975414            0.182305          0.182971             

In [62]:
# --- cache a normalized mean IG map for cross-model comparison ---
global IG_CLEAN_MEANS
try:
    IG_CLEAN_MEANS
except NameError:
    IG_CLEAN_MEANS = {}

mean_map = Ec.abs().mean(dim=0)              # (C, T)
vec = mean_map.flatten().detach().cpu().numpy()
vec = vec / (np.linalg.norm(vec) + 1e-12)    # normalize for cosine
IG_CLEAN_MEANS[(model_name, seed)] = vec


In [63]:
# --- cross-model cosine on clean IG maps (pick a seed) ---
seed_pick = 42
keys = [(m, s) for (m, s) in IG_CLEAN_MEANS.keys() if s == seed_pick]
if keys:
    mats = np.stack([IG_CLEAN_MEANS[k] for k in keys], axis=0)
    cos = mats @ mats.T
    labels = [k[0] for k in keys]
    print(f"\nCosine similarity (clean IG mean maps), seed={seed_pick}:")
    print(pd.DataFrame(cos, index=labels, columns=labels))
else:
    print(f"[IG sanity] No cached IG maps for seed {seed_pick}.")


[IG sanity] No cached IG maps for seed 42.


In [64]:
IDX_CAP = slice(0, IG_INT_BS)   # e.g., 8


In [58]:
with torch.no_grad():
    feats_clean = model.features(X)  # shape (N, D)
    feats_adv   = model.features(X_adv)
np.save(f"{base}/feats_clean_eps{eps}.npy", feats_clean.cpu().numpy())
np.save(f"{base}/feats_adv_eps{eps}.npy", feats_adv.cpu().numpy())


AttributeError: 'CTNet' object has no attribute 'features'

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))
tsne = TSNE(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))


[main 263a434] attacks s1
 2 files changed, 682 insertions(+)
 create mode 100644 results/adversarial_results_1.csv
 create mode 100644 results/adversarial_results_MASTER.csv
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 12 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 26.75 KiB | 4.46 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/VictoryChianumba/robust-eeg-models.git
   78a528d..263a434  main -> main


# Single attack runner for PGD